# 6. Naming your formulas

Two loose ends from earlier notebooks come together here.

In [notebook 3](03-finding-repeated-phrases.ipynb), we found that the software's raw candidate phrases often come back as several overlapping, shifted fragments of one longer formula, and we pieced them back together *by hand*. In [notebook 5](05-where-do-documents-begin-and-end.ipynb), we saw that this corpus contains several distinct *kinds* of formula &mdash; openers for letters, openers for petitions, decision formulas, attendance-list formulas &mdash; and that organising them by kind is what actually makes a boundary-finding (or any other) approach work well.

This notebook gives you tools for both: doing the fragment-piecing automatically, and organising the results into named categories of your own choosing, with example text to browse as you go.

In [1]:
import csv
import gzip
import re
from collections import Counter

TOKEN_PATTERN = re.compile(r"[A-Za-z\u00c0-\u00ff]+")


def tokenize(text):
    return TOKEN_PATTERN.findall(text.lower())


rows = []
with gzip.open('data/resolution_paragraphs/paragraphs-3823.tsv.gz', 'rt', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter='\t'):
        rows.append(row)

documents = [tokenize(row['text']) for row in rows]
flat_stream = [token for doc in documents for token in doc]
print(f'{len(documents)} documents, {len(flat_stream)} tokens')

2909 documents, 449905 tokens


## Part 1: piecing fragments together automatically

Let's rediscover candidate phrases exactly as we did in notebook 3, and look at the top results again.

In [2]:
from formula_detection.search import FormulaSearch

formula_search = FormulaSearch(documents, min_term_freq=10, min_cooc_freq=10, skip_size=2)

phrase_freq = Counter()
for match in formula_search.extract_phrases('sub_phrases', min_phrase_length=3, max_phrase_length=6,
                                            min_neighbour_cooc=1):
    phrase_freq[match.candidate_phrase.phrase_string] += 1

candidate_phrases = [phrase for phrase, _ in phrase_freq.most_common(40)]
print('Top 6:')
for phrase in candidate_phrases[:6]:
    print(f'  {phrase_freq[phrase]: >4}  {phrase}')

1. Iterating over sentences to calculate term frequencies
    full collection size (tokens): 449,905
    full lexicon size (types): 21,185
    minimum term frequency: 10
    minimum frequency lexicon size: 2,880
2. Iterating over sentences to calculate the co-occurrence frequencies


docs: 2,909	num_words: 449,905	num_coocs: 1,125,317	num distinct coocs: 182,121
    co-occurrence index size: 182,121
Minimum co-occurrence frequency: 10


Top 6:
   839  op gedelibereert zynde is goedgevonden en
   839  gedelibereert zynde is goedgevonden en verstaan
   836  waar op gedelibereert zynde is goedgevonden
   835  waar op geen resolutie is gevallen
   720  advertentie waar op geen resolutie is
   718  houdende advertentie waar op geen resolutie


There they are again: the same overlapping fragments of the `waar op gedelibereert zynde is goedgevonden en verstaan dat...` formula we reconstructed by hand in notebook 3. We first need to know *where* each candidate occurs, not just how often, which is what `PhraseCooccurrenceIndex` gives us.

In [3]:
from formula_detection.phrase_cooccurrence import PhraseCooccurrenceIndex

cooc_index = PhraseCooccurrenceIndex(candidate_phrases, windows=[5])
cooc_index.index_stream(flat_stream)

Now for the automatic version of what we did by hand. `merge_overlapping_phrases` looks for candidate phrases that almost always occur within a few tokens of each other, every single time &mdash; the signature of overlapping slices of one formula, rather than two genuinely different formulas that just happen to be related. It groups those into one cluster and picks a single representative ("canonical") phrase for each cluster. **You don't need to understand its internals to use it** &mdash; it's built on a graph-based technique called *community detection*, the same one used for the more advanced document-pattern analysis covered in `Document-Pattern-Detection.ipynb`, just applied here at a much smaller scale.

In [4]:
from formula_detection.motif import merge_overlapping_phrases, merge_phrase_positions

phrase_to_canonical = merge_overlapping_phrases(cooc_index.phrase_positions, window=5,
                                                min_support=5, min_lift=3.0)
merged_positions = merge_phrase_positions(cooc_index.phrase_positions, phrase_to_canonical,
                                          dedup_distance=3)

print(f'{len(candidate_phrases)} raw candidates -> {len(merged_positions)} canonical phrases')

40 raw candidates -> 9 canonical phrases


From 40 raw, overlapping candidates down to 9 canonical phrases &mdash; a much more manageable number to actually look at and organise. Let's see what they are, and check whether the `gedelibereert` formula came back together as we'd hope.

In [5]:
for canonical, positions in sorted(merged_positions.items(), key=lambda kv: -len(kv[1])):
    members = [phrase for phrase, c in phrase_to_canonical.items() if c == canonical]
    print(f'{len(positions): >5}  {canonical!r}')
    print(f'        merged from {len(members)} raw candidate(s)')

 1297  'waar op geen resolutie is gevallen'
        merged from 7 raw candidate(s)
 1222  'gedelibereert zynde is goedgevonden en verstaan'
        merged from 8 raw candidate(s)
  698  'ook geresumeert en gearresteert zyn de'
        merged from 8 raw candidate(s)
  671  'haar hoog mog resolutie van den'
        merged from 3 raw candidate(s)
  513  'ontfangen een missive van den heere'
        merged from 2 raw candidate(s)
  375  'van sijne majesteit den koning van'
        merged from 4 raw candidate(s)
  338  'is ter vergaderinge geleesen de requeste'
        merged from 2 raw candidate(s)
  287  'extraordinaris gedeputeerde uit de provincie van'
        merged from 3 raw candidate(s)
  245  'haar hoog mog gedeputeerden tot de'
        merged from 3 raw candidate(s)


One detail worth understanding: the "canonical" phrase for a cluster is simply the single most frequent raw fragment in it &mdash; it's a representative label, *not* the full reconstructed formula. To actually see the complete formula, use `concordance` (from [notebook 4](04-old-spelling-same-phrase.ipynb)'s companion module) on the canonical phrase's *merged* positions, which now correctly combine occurrences from every fragment in its cluster.

In [6]:
from formula_detection.phrase_typing import concordance

decision_canonical = [c for c in merged_positions if 'gedelibereert' in c][0]
for line in concordance(decision_canonical, flat_stream, positions=merged_positions[decision_canonical],
                        window=6, max_examples=3):
    print(' ', line)

  voorleede maand houdende een nieuwejaars wensch [waar op gedelibereert zynde is goedgevonden] en verstaan dat de voorsz missive
  handen gestelt waar op gedelibereert zynde [is goedgevonden en verstaan dat copie] van de voorschreeve requeste en bylaage
  ambassadeur daar van kennisse geeven soude [waar op gedelibereert zynde is goedgevonden] en verstaan dat copie van de


There's the full formula, exactly as we reconstructed it by hand in notebook 3, now recovered automatically.

One word of caution, since we've already seen this lesson apply elsewhere in this series: `window=5` here is small on purpose, to only catch slices that are *almost always* this close together. Pick too large a window and you risk merging two genuinely different formulas that simply tend to follow one another in the same short paragraph (we found exactly this happening with `window=10` on this corpus, before narrowing it down) &mdash; the same kind of trade-off we met when choosing a similarity threshold in notebook 4. There's no setting that's risk-free; always check what got merged, the way we just did.

## Part 2: organising formulas into named categories

We now have 9 clean, distinct candidate formulas instead of 40 confusing fragments &mdash; few enough to organise by hand, with the toolkit's help. The **phrase typing workbench** lets you:

* browse real examples of a phrase before deciding what it is,
* assign it to one or more categories you define yourself (a phrase can belong to more than one),
* and get ranked suggestions for which of the *remaining* phrases probably belong to a category you've already started.

Let's set it up, using the 9 canonical phrases from Part 1.

In [7]:
from formula_detection.phrase_typing import PhraseTypeRegistry, TypeSuggester
from formula_detection.phrase_typing_ui import PhraseTypingApp

canonical_phrases = list(merged_positions.keys())
phrase_frequencies = {phrase: len(positions) for phrase, positions in merged_positions.items()}

registry = PhraseTypeRegistry()
suggester = TypeSuggester(merged_positions, flat_stream, context_window=10)

app = PhraseTypingApp(registry, suggester, canonical_phrases, phrase_frequencies)
app.display()

If you're running this notebook yourself, try it now: pick `gedelibereert zynde is goedgevonden en verstaan` in the *Browse & assign* tab, read a few examples, and assign it to a new category &mdash; `decision formula` is a reasonable name, since that's what it is. Do the same for `ontfangen een missive van den heere` as an `opening formula`. Then switch to the *Suggestions* tab and see what the tool proposes for each category among the phrases you haven't categorised yet.

## The same thing without clicking through the widget

Everything the widget does is also directly callable in code, which is useful for following along here, or for scripting this process over many phrases at once.

In [8]:
# Look up the canonical phrases by what they contain, rather than hardcoding the
# exact string: when two raw candidates tie in frequency (as happens here for the
# decision formula), which one becomes the canonical representative can vary between
# runs, so a hardcoded literal string isn't reliable here.
decision_phrase = [c for c in canonical_phrases if 'gedelibereert' in c][0]
no_action_phrase = [c for c in canonical_phrases if 'geen resolutie is gevallen' in c][0]
missive_opener = [c for c in canonical_phrases if 'ontfangen een missive' in c][0]
requeste_opener = [c for c in canonical_phrases if 'requeste' in c][0]

scripted_registry = PhraseTypeRegistry()
scripted_registry.assign(decision_phrase, 'decision formula')
scripted_registry.assign(no_action_phrase, 'decision formula')
scripted_registry.assign(missive_opener, 'opening formula')
scripted_registry.assign(requeste_opener, 'opening formula')

print(scripted_registry.summary())

decision formula (2): ['gedelibereert zynde is goedgevonden en verstaan', 'waar op geen resolutie is gevallen']
opening formula (2): ['is ter vergaderinge geleesen de requeste', 'ontfangen een missive van den heere']


In [9]:
uncategorised = scripted_registry.uncategorised(canonical_phrases)
print('still uncategorised:', uncategorised)
print()

for type_name, window in [('decision formula', 30), ('opening formula', 50)]:
    members = scripted_registry.types[type_name].members
    suggestions = suggester.suggest_for_type(members, uncategorised, window=window, top_n=5)
    print(f'suggestions for {type_name!r} (window={window}):')
    for phrase, affinity, similarity in suggestions:
        print(f'  affinity={affinity:.2f}  context_sim={similarity:.2f}  {phrase}')
    print()

still uncategorised: ['van sijne majesteit den koning van', 'ook geresumeert en gearresteert zyn de', 'haar hoog mog resolutie van den', 'extraordinaris gedeputeerde uit de provincie van', 'haar hoog mog gedeputeerden tot de']

suggestions for 'decision formula' (window=30):
  affinity=0.84  context_sim=0.70  van sijne majesteit den koning van
  affinity=0.56  context_sim=0.56  haar hoog mog gedeputeerden tot de
  affinity=0.11  context_sim=0.52  ook geresumeert en gearresteert zyn de
  affinity=0.09  context_sim=0.72  haar hoog mog resolutie van den
  affinity=0.02  context_sim=0.56  extraordinaris gedeputeerde uit de provincie van

suggestions for 'opening formula' (window=50):
  affinity=0.79  context_sim=0.64  van sijne majesteit den koning van
  affinity=0.70  context_sim=0.47  ook geresumeert en gearresteert zyn de
  affinity=0.39  context_sim=0.51  extraordinaris gedeputeerde uit de provincie van
  affinity=0.18  context_sim=0.50  haar hoog mog gedeputeerden tot de
  affinity=0.

`van sijne majesteit den koning van` ("of His Majesty the King of...") scores fairly high for *both* categories &mdash; which is a useful result to sit with rather than accept automatically. It's a title phrase that can appear near either a letter's opening *or* its decision, simply because these resolutions are short and many different phrases end up close together. A high suggestion score means "this often occurs near phrases you've already confirmed", not "this is definitely the same kind of formula" &mdash; the same caution from notebook 4 applies again here: **browse the actual examples (the *Browse & assign* tab, or `concordance` directly) before accepting a suggestion**, especially for borderline cases like this one.

## Next steps

We now have the tools to discover candidate formulas, consolidate their fragments, handle spelling variation, find document boundaries, and organise everything into named categories. [Notebook 7](07-comparing-across-time-or-place.ipynb), the last in this series, puts all of this together to answer the kind of question that probably motivated you to start: has formulaic language use changed &mdash; over time, between archives, or between scribes?